# MANTIS V2 × XMAD-Bench: Strategy 3 (Multi-Scale Temporal Chunking)
### End-to-End GPU Cloud Pipeline on Kaggle (T4 GPU Accelerator)

**Objective**: Resolve the Valle-X neural codec downsampling bottleneck by extracting multi-resolution temporal features ($512, 1024, 2048$ points $\to 768$-d representation) and training an ArcMargin hypersphere metric probe ($m=0.15, s=24.0$).

> **Prerequisites on Kaggle**:
> 1. Notebook Settings -> **Accelerator: GPU T4 x2** (or GPU P100).
> 2. Notebook Settings -> **Internet: On** (required to stream from Hugging Face).

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

In [ ]:
# 2. Install Dependencies
!pip install -q datasets mantis-foundation soundfile librosa scikit-learn pandas matplotlib pyarrow

In [ ]:
# 3. Core Architecture & Utility Setup
import os, sys, time, json, math, glob, io, wave
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from datasets import load_dataset, Audio
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_OUTPUT_DIR = '/kaggle/working/xmad_mantis_multiscale'
SHARDS_DIR = os.path.join(BASE_OUTPUT_DIR, 'shards')
RESULTS_DIR = os.path.join(BASE_OUTPUT_DIR, 'results')
os.makedirs(SHARDS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def parse_wav_bytes(audio_bytes):
    with io.BytesIO(audio_bytes) as bio:
        with wave.open(bio, 'rb') as wf:
            n_channels = wf.getnchannels()
            sampwidth = wf.getsampwidth()
            n_frames = wf.getnframes()
            raw_data = wf.readframes(n_frames)
            if sampwidth == 2:
                audio_arr = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
            elif sampwidth == 4:
                audio_arr = np.frombuffer(raw_data, dtype=np.int32).astype(np.float32) / 2147483648.0
            else:
                audio_arr = np.frombuffer(raw_data, dtype=np.uint8).astype(np.float32) / 128.0 - 1.0
            if n_channels > 1:
                audio_arr = audio_arr.reshape(-1, n_channels).mean(axis=1)
    return audio_arr

print('Loading MANTIS V2 foundation model onto', DEVICE)
network = MantisV2(device=DEVICE).from_pretrained('paris-noah/MantisV2')
trainer = MantisTrainer(device=DEVICE, network=network)
print('MANTIS V2 ready for streaming extraction!')

In [ ]:
# 4. High-Throughput Streaming Feature Extraction (Multi-Scale: 512, 1024, 2048 -> 768-d)
def process_shard_gpu(shard_idx):
    shard_str = f'{shard_idx:05d}'
    shard_path = f'data/test-{shard_str}-of-00200.parquet'
    out_file = os.path.join(SHARDS_DIR, f'shard_{shard_str}.parquet')
    if os.path.exists(out_file):
        return out_file

    ds = load_dataset('SpeechAntiSpoofingBenchmarks/XMAD', data_files={'test': [shard_path]}, split='test', streaming=True)
    ds = ds.cast_column('audio', Audio(decode=False))

    records, x512, x1024, x2048 = [], [], [], []
    for sample in ds:
        audio_dict = sample.get('audio')
        if not audio_dict or not audio_dict.get('bytes'):
            continue
        try:
            wav = parse_wav_bytes(audio_dict['bytes'])
        except Exception:
            continue
        if len(wav) < 512:
            continue

        t = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        r512 = F.interpolate(t, size=512, mode='linear', align_corners=False).squeeze(0).numpy()
        r1024 = F.interpolate(t, size=1024, mode='linear', align_corners=False).squeeze(0).numpy()
        r2048 = F.interpolate(t, size=2048, mode='linear', align_corners=False).squeeze(0).numpy()

        x512.append(r512); x1024.append(r1024); x2048.append(r2048)
        meta = json.loads(sample.get('notes', '{}'))
        records.append({
            'utterance_id': meta.get('utterance_id', sample.get('path', '')),
            'label': int(sample.get('label', 0)),
            'attack': meta.get('attack', 'bonafide' if sample.get('label') == 0 else 'unknown'),
            'speaker_id': meta.get('speaker_id', 'unknown'),
            'language': meta.get('language', ''),
            'duration_sec': float(len(wav) / 16000.0),
            'original_split': meta.get('split', 'test')
        })

    if not records:
        return None

    X512, X1024, X2048 = np.array(x512, dtype=np.float32), np.array(x1024, dtype=np.float32), np.array(x2048, dtype=np.float32)
    batch_size = 128
    z512, z1024, z2048 = [], [], []
    for b in range(0, len(X512), batch_size):
        z512.append(trainer.transform(X512[b:b+batch_size]))
        z1024.append(trainer.transform(X1024[b:b+batch_size]))
        z2048.append(trainer.transform(X2048[b:b+batch_size]))

    Z_multi = np.concatenate([np.concatenate(z512), np.concatenate(z1024), np.concatenate(z2048)], axis=1)
    df = pd.DataFrame(records)
    df['embedding'] = [v.tolist() for v in Z_multi]
    df.to_parquet(out_file, index=False)
    return out_file

# Process a benchmark subset (e.g. 20 representative shards including Valle-X shards 106, 113, 191)
target_shards = [0, 1, 2, 54, 55, 81, 82, 106, 113, 150, 151, 191]
print(f'Starting extraction of {len(target_shards)} benchmark shards...')
t_start = time.time()
for s_idx in target_shards:
    t0 = time.time()
    res = process_shard_gpu(s_idx)
    print(f'Shard {s_idx:05d} finished in {time.time()-t0:.1f}s -> {res}')
print(f'All shards extracted in {(time.time()-t_start)/60:.2f} mins!')

In [ ]:
# 5. Metric Learning Architecture (ArcMargin Head on 768-d Multi-Scale Embeddings)
class MultiScaleMetricProbe(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=256, out_dim=64, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, out_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=1)

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features=64, out_features=2, s=24.0, m=0.15):
        super().__init__()
        self.s = s; self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, input_feat, label=None):
        w = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(input_feat, w).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        if label is None:
            return cosine * self.s, cosine
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2)).clamp(0, 1)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1.0)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s, cosine

class MultiScaleAntiSpoofingSystem(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=256, out_dim=64, s=24.0, m=0.15):
        super().__init__()
        self.probe = MultiScaleMetricProbe(in_dim=in_dim, hidden_dim=hidden_dim, out_dim=out_dim)
        self.arc = ArcMarginProduct(in_features=out_dim, out_features=2, s=s, m=m)
    def forward(self, x, label=None):
        feat = self.probe(x)
        return self.arc(feat, label)
    @torch.no_grad()
    def predict_score(self, x):
        self.eval()
        feat = self.probe(x)
        logits, _ = self.arc(feat, label=None)
        return F.softmax(logits, dim=1)[:, 1].cpu().numpy()

In [ ]:
# 6. Training & Evaluation Pipeline
def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0), float(thresholds[idx])

shard_files = sorted(glob.glob(os.path.join(SHARDS_DIR, 'shard_*.parquet')))
df = pd.concat([pd.read_parquet(f) for f in shard_files], ignore_index=True)
print(f'Total samples: {len(df):,} | Real: {(df.label==0).sum():,} | Fake: {(df.label==1).sum():,}')
print(df.attack.value_counts())

# LOAO Evaluation focused on Valle-X vs Vocoders
top_attacks = ['vallex', 'xtts_v2', 'vits+openvoice', 'glowtts+freevc']
loao_results = []

for held_out in top_attacks:
    fake_tr = df[(df.label == 1) & (df.attack != held_out)]
    fake_te = df[(df.label == 1) & (df.attack == held_out)]
    if len(fake_te) < 10: continue

    real_spks = sorted(df[df.label == 0].speaker_id.unique())
    half = len(real_spks) // 2
    real_tr = df[(df.label == 0) & (df.speaker_id.isin(real_spks[:half]))]
    real_te = df[(df.label == 0) & (df.speaker_id.isin(real_spks[half:]))]

    n_tr = min(3000, len(fake_tr), len(real_tr))
    n_te = min(500, len(fake_te), len(real_te))

    tr_sub = pd.concat([real_tr.sample(n_tr, random_state=42), fake_tr.sample(n_tr, random_state=42)])
    te_sub = pd.concat([real_te.sample(n_te, random_state=42), fake_te.sample(n_te, random_state=42)])

    X_tr, y_tr = np.vstack(tr_sub.embedding.values), tr_sub.label.values
    X_te, y_te = np.vstack(te_sub.embedding.values), te_sub.label.values

    model = MultiScaleAntiSpoofingSystem(in_dim=768, s=24.0, m=0.15).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=128, shuffle=True)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(15):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx, by)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()

    probs = model.predict_score(torch.tensor(X_te, dtype=torch.float32).to(DEVICE))
    acc = accuracy_score(y_te, (probs >= 0.5).astype(int)) * 100
    auc = roc_auc_score(y_te, probs) * 100
    eer, _ = compute_eer(y_te, probs)
    eer *= 100
    print(f'>> [LOAO Held-Out: {held_out:<16}] ACC: {acc:6.2f}% | AUC: {auc:6.2f}% | EER: {eer:6.2f}%')
    loao_results.append({'attack': held_out, 'acc': acc, 'auc': auc, 'eer': eer})

res_df = pd.DataFrame(loao_results)
res_df.to_csv(os.path.join(RESULTS_DIR, 'loao_multiscale_results.csv'), index=False)
print('Finished LOAO evaluation!')
print(res_df)